# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Dawngend/FlyRank-Machine-Learning-Internship/blob/main/work/notebooks/w01_research_question.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

**Chosen Lane:** Lane 2 — Refresh / Content Opportunity Scoring (Core Lane)

**Why this lane:**
Organic content decay is one of the most critical operational challenges in search engine optimization (SEO) and content management. Websites often host thousands of published articles, but editorial teams possess strictly finite time and budget. Without a principled data-driven prioritization framework, teams waste bandwidth updating healthy or unpromising articles while high-potential articles experiencing traffic decay slip through unaddressed.

This lane focuses on building a ranked review queue backed by predictive risk scores and transparent reason codes (e.g., `stale_visible_page`, `declining_with_demand`, `low_ctr_visible_page`). By combining multi-signal search performance with machine learning models, we can direct limited editorial review capacity toward the pages with the highest expected impact.

In [2]:
# Quick verification of lane choice and dataset availability
from pathlib import Path

raw_path = Path('../../data/raw/content_refresh_anonymized.csv')
if not raw_path.exists():
    raw_path = Path('data/raw/content_refresh_anonymized.csv')

print('[✓] Lane Selected: Refresh / Content Opportunity Scoring (Lane 2)')
print(f'[✓] Starter Dataset Path: {raw_path.resolve()}')
print(f'[✓] File Exists: {raw_path.exists()}')


[✓] Lane Selected: Refresh / Content Opportunity Scoring (Lane 2)
[✓] Starter Dataset Path: D:\Flyrank\FlyRank-Machine-Learning-Internship\data\raw\content_refresh_anonymized.csv
[✓] File Exists: True


## 2. The question: decision, action, cost of a wrong call

### Frame Summary
- **Research Question:** *"Which content items (pages) should an editorial team review first for refresh, expansion, or protection to prevent organic traffic decay?"*
- **Unit of Analysis:** A single pseudonymized content item (`content_id`) evaluated over a trailing 90-day window.
- **Decision Improved:** Deciding which specific pages out of thousands of published articles deserve immediate editorial review and resource allocation.
- **Who Acts & Action Taken:** Content Editors, SEO Strategists, and Content Managers. Based on the ranked queue and transparent reason codes, they perform targeted interventions:
  - Updating outdated facts/statistics (`stale_visible_page`)
  - Rewriting underperforming title tags/meta descriptions (`low_ctr_visible_page`)
  - Expanding thin content (`thin_visible_page`)
  - Protecting Page-1 decaying assets (`declining_with_demand`)
- **Cost of a Wrong Call:**
  - **False Positive (flagging a healthy/recovering page):** Wastes 2–5 hours of editorial time per article rewrite and risks disrupting existing search rankings.
  - **False Negative (missing a decaying high-demand page):** Results in persistent compounding traffic loss, lost organic conversions, and competitor takeover of key search positions.
- **Why Data & ML Help:** Simple heuristics (e.g. age > 180 days) fail to capture non-linear, multi-signal interactions between search volume, position drift, CTR, and user engagement across diverse client domains. ML models can learn subtle multi-signal decay patterns to prioritize candidates far more accurately than hand-written rules.

In [3]:
# Formal definition of decision parameters
decision_params = {
    'Target User': 'Content Editor / SEO Strategist',
    'Unit of Analysis': 'content_id (pseudonymized content item)',
    'Primary Decision': 'Which page to review and refresh first',
    'Primary Action': 'Editorial refresh, metadata rewrite, or content expansion',
    'Key Evaluation Metric': 'Precision@K (Precision@50 on top-ranked queue)'
}

for key, val in decision_params.items():
    print(f'{key:22s}: {val}')


Target User           : Content Editor / SEO Strategist
Unit of Analysis      : content_id (pseudonymized content item)
Primary Decision      : Which page to review and refresh first
Primary Action        : Editorial refresh, metadata rewrite, or content expansion
Key Evaluation Metric : Precision@K (Precision@50 on top-ranked queue)


## 3. Quick look at the data (2-3 real numbers)

Analysis of the starter dataset (`data/raw/content_refresh_anonymized.csv`) reveals key empirical numbers that justify prioritizing this lane:

1. **Scale & Decay Prevalence:** The dataset contains **30,000 content items** across **32 pseudonymized clients**. A staggering **16,262 items (54.21%)** show a downward traffic trend (`trend_direction == "down"`), demonstrating that content decay is widespread.
2. **High-Demand Opportunity Pool:** **16,726 items (55.75%)** generate at least 500 impressions over 90 days (`impressions_90d >= 500`), establishing a substantial pool of high-visibility assets where refreshes can drive meaningful traffic recovery.
3. **Low-CTR Quick Wins:** **9,759 items (32.53%)** rank on Page 1 or 2 (`avg_position` between 1 and 20) with high demand (`impressions_90d >= 500`) but suffer from low click-through rates (`ctr < 0.5%`), revealing an immediate metadata optimization opportunity.
4. **Baseline vs. ML Model Lift:** In client-holdout evaluations, a hand-written baseline rule achieves a **Precision@50 of 0.240** (12/50 correct), while a Random Forest model achieves **0.680** (34/50 correct) — proving a **2.83x performance gain** over naive rules.

In [4]:
import pandas as pd
import json
from pathlib import Path

# 1. Load starter raw dataset
data_path = Path('../../data/raw/content_refresh_anonymized.csv')
if not data_path.exists():
    data_path = Path('data/raw/content_refresh_anonymized.csv')

df = pd.read_csv(data_path)

total_rows = len(df)
total_clients = df['client_id'].nunique()
declining_count = (df['trend_direction'] == 'down').sum()
declining_pct = (declining_count / total_rows) * 100

high_imp_count = (df['impressions_90d'] >= 500).sum()
high_imp_pct = (high_imp_count / total_rows) * 100

low_ctr_opps = ((df['impressions_90d'] >= 500) & (df['avg_position'] > 0) & (df['avg_position'] <= 20) & (df['ctr'] < 0.5)).sum()

print('=' * 65)
print('STARTER DATASET EMPIRICAL EVIDENCE')
print('=' * 65)
print(f'1. Inventory Scale        : {total_rows:,} content items across {total_clients} clients')
print(f'2. Decaying Content Pool   : {declining_count:,} items ({declining_pct:.2f}%) exhibiting downward trend')
print(f'3. High-Demand Items      : {high_imp_count:,} items ({high_imp_pct:.2f}%) with >= 500 90d impressions')
print(f'4. Low-CTR Quick-Wins     : {low_ctr_opps:,} high-visibility Page 1/2 items with CTR < 0.5%')

# 2. Load model benchmark results
results_path = Path('../../outputs/model_results.json')
if not results_path.exists():
    results_path = Path('outputs/model_results.json')

if results_path.exists():
    with open(results_path) as f:
        results = json.load(f)
    print('\n' + '=' * 65)
    print('MODEL BENCHMARK RESULTS (Client-Holdout Split)')
    print('=' * 65)
    if 'baseline' in results:
        b_metrics = results['baseline']
        print(f" - {'baseline_rules':22s}: Precision@50 = {b_metrics.get('baseline_precision_at_50', 0):.3f} | ROC-AUC = {b_metrics.get('baseline_roc_auc', 0):.3f}")
    if 'models' in results:
        for model_name, metrics in results['models'].items():
            prec50 = metrics.get('precision_at_50', 0)
            roc_auc = metrics.get('roc_auc', 0)
            print(f" - {model_name:22s}: Precision@50 = {prec50:.3f} | ROC-AUC = {roc_auc:.3f}")


STARTER DATASET EMPIRICAL EVIDENCE
1. Inventory Scale        : 30,000 content items across 32 clients
2. Decaying Content Pool   : 16,262 items (54.21%) exhibiting downward trend
3. High-Demand Items      : 16,726 items (55.75%) with >= 500 90d impressions
4. Low-CTR Quick-Wins     : 9,759 high-visibility Page 1/2 items with CTR < 0.5%

MODEL BENCHMARK RESULTS (Client-Holdout Split)
 - baseline_rules        : Precision@50 = 0.240 | ROC-AUC = 0.627
 - decision_tree         : Precision@50 = 0.620 | ROC-AUC = 0.742
 - logistic_regression   : Precision@50 = 0.400 | ROC-AUC = 0.700
 - random_forest         : Precision@50 = 0.680 | ROC-AUC = 0.747


## 4. Careful words: what I can and can't claim

### What I CAN claim:
- **Observed Associations:** Quantifiable statistical relationships between historical search/engagement signals (age, impressions, average position, CTR) and performance trends.
- **Relative Priority Scoring:** An empirical ranking that prioritizes candidate pages for editorial review more effectively than random selection or naive rules.
- **Decision-Support Value:** Demonstrable performance lift (Precision@50) on unseen client holdout data to optimize human editorial workflows.

### What I CANNOT claim:
- **Causal Proof:** That refreshing a flagged page *will cause* traffic to recover (proving causality requires controlled A/B experiments).
- **Google Algorithm Decoding:** That this project uncovers, models, or predicts Google's proprietary search engine ranking algorithm.
- **Algorithmic Penalty Diagnosis:** That a traffic decline is definitively caused by a Google penalty rather than macro search trends, competitor actions, or seasonality.

In [5]:
# Formal claim boundaries verification
allowed_claims = [
    'Observed historical correlations between signals and decay',
    'Relative ranking of refresh priority candidates',
    'Decision-support to optimize editorial reviewer time'
]

forbidden_claims = [
    'Causal proof that a refresh guarantees traffic recovery',
    'Reverse engineering of Google search algorithms',
    'Definitive attribution of decline to Google penalties'
]

print('[✓] ALLOWED CLAIMS:')
for claim in allowed_claims:
    print(f'   - {claim}')

print('\n[X] FORBIDDEN CLAIMS:')
for claim in forbidden_claims:
    print(f'   - {claim}')


[✓] ALLOWED CLAIMS:
   - Observed historical correlations between signals and decay
   - Relative ranking of refresh priority candidates
   - Decision-support to optimize editorial reviewer time

[X] FORBIDDEN CLAIMS:
   - Causal proof that a refresh guarantees traffic recovery
   - Reverse engineering of Google search algorithms
   - Definitive attribution of decline to Google penalties


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.